# ECC KV Cache — A100 80GB Session
**Run cells top to bottom. On any restart, re-run Cell 1 only.**

In [ ]:
# CELL 1 — Run every restart
import os, sys, torch, warnings
warnings.filterwarnings('ignore')  # suppress torch_dtype deprecation

# 1. Clone repo if VM was fully reset, otherwise pull latest fixes
if not os.path.exists('/content/ecc-kv-cache'):
    os.system('git clone https://github.com/j-arndt/ecc-kv-cache /content/ecc-kv-cache')
else:
    os.system('git -C /content/ecc-kv-cache pull origin main')

os.chdir('/content/ecc-kv-cache')
os.makedirs('results', exist_ok=True)

# 2. Tell system linker where torch libs are (fixes libc10.so not found)
torch_lib = torch.__path__[0] + '/lib'
os.system(f'echo {torch_lib} > /etc/ld.so.conf.d/torch.conf && ldconfig')

# 3. Clean stale build artifacts (avoids ABI mismatch across torch versions)
os.system('rm -rf build/ custom_ecc_cuda*.so')
os.system('python setup.py build_ext --inplace 2>&1 | grep -E "building|copying|error"')

# 4. Make custom_kv importable by all !python subprocesses
sys.path.insert(0, '/content/ecc-kv-cache')
os.environ['PYTHONPATH'] = '/content/ecc-kv-cache'

# 5. Verify extension loads
import importlib
import custom_ecc_cuda
print('CUDA extension:', list(filter(lambda x: not x.startswith('_'), dir(custom_ecc_cuda))))
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM total:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('\n✓ Cell 1 complete — proceed to Cell 2')


In [ ]:
# CELL 2 — CPU unit tests (28 tests, ~7 seconds)
!python -m pytest tests/unit/ -v --tb=short


In [ ]:
# CELL 3 — HuggingFace login (once per VM session)
from google.colab import userdata
from huggingface_hub import login
import os

token = userdata.get('hf_token2')  # matches your Colab secret name
os.environ['HF_TOKEN'] = token
login(token=token)
print('\n✓ Logged in')


In [ ]:
# CELL 4 — Calibration (~10 min). Skips if already done this session.
import os

if os.path.exists('calibration_config.json'):
    import json
    cfg = json.load(open('calibration_config.json'))
    n = len(cfg.get('layers', {}))
    print(f'Calibration already done: {n} layers. Skipping.')
    print('Delete calibration_config.json to re-run.')
else:
    !python scripts/run_calibration.py \
        --model meta-llama/Meta-Llama-3-8B-Instruct \
        --n-samples 512 \
        --output calibration_config.json
    import json
    cfg = json.load(open('calibration_config.json'))
    print(f'\n✓ Calibrated {len(cfg["layers"])} layers')


In [ ]:
# CELL 5 — Smoke test (~5 min). Verifies ECC cache works end-to-end.
!python scripts/smoke_test.py \
    --model meta-llama/Meta-Llama-3-8B-Instruct \
    --ctx-len 2000 \
    --n-tokens 50


In [ ]:
# CELL 6 — VRAM benchmark (~20 min)
!python benchmarks/vram_reduction.py \
    --model meta-llama/Meta-Llama-3-8B-Instruct \
    --ctx-lengths 8000 32000 64000 128000 \
    --output results/vram_results.json


In [ ]:
# CELL 7 — Throughput benchmark (~15 min)
!python benchmarks/throughput.py \
    --model meta-llama/Meta-Llama-3-8B-Instruct \
    --ctx-len 64000 \
    --n-tokens 200 \
    --output results/throughput.json


In [ ]:
# CELL 8 — NIAH benchmark (~4.5 hours, 100 trials)
# Crash-safe: if interrupted, re-run this cell and it resumes automatically.
!python benchmarks/niah_128k.py \
    --model meta-llama/Meta-Llama-3-8B-Instruct \
    --ctx-lengths 8000 32000 64000 128000 \
    --depths 0.0 0.25 0.5 0.75 1.0 \
    --trials 100 \
    --methods fp16 int4_bnb int4_ecc \
    --output results/niah_results.json


In [ ]:
# CELL 9 — Generate comparison table from results
!python benchmarks/compare_baselines.py \
    --vram results/vram_results.json \
    --niah results/niah_results.json \
    --throughput results/throughput.json \
    --output results/comparison_table.md


In [ ]:
# CELL 10 — Push all results to GitHub
!git config user.email 'j-arndt@users.noreply.github.com'
!git config user.name 'j-arndt'
!git add results/ calibration_config.json
!git commit -m 'results: A100 80GB benchmark — VRAM, NIAH 100-trial, throughput'
!git tag v0.1.0 --force
!git push origin main --tags
print('\n✓ Done! Results at https://github.com/j-arndt/ecc-kv-cache')
